## Initialize Qdrant

In [3]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

In [9]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient, models 
from langchain_openai import OpenAIEmbeddings
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")


dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
qdrant = QdrantVectorStore.from_existing_collection(
    collection_name="document_collection",
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode = RetrievalMode.HYBRID,
    vector_name = "dense",
    sparse_vector_name = "sparse"
    )

In [11]:
qdrant.similarity_search("What is Home Credit?, k=1")

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-02T00:41:51+00:00', 'source': '/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/tmppep5_hzk.pdf', 'file_path': '/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/tmppep5_hzk.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-02-02T00:41:51+00:00', 'trapped': '', 'modDate': 'D:20260202004151Z', 'creationDate': 'D:20260202004151Z', 'page': 0, '_id': 'f5f538b7-78e5-4512-9572-dd1db0cae57b', '_collection_name': 'document_collection'}, page_content='Khoa Bui\n\x83 (437) 443-7831\n# bndk2108@gmail.com\nï linkedin.com/in/bndk2108\n§ github.com/hyolee1999\nEducation\nHo Chi Minh University of Technology\nOct 2017 – April 2022\nBachelor of Computer Science\nHo Chi Minh City, Vietnam\n• Thesis: Develop Android Application for Removing Unwanted Objects. York University\nSep 2025 – Present\nMaster of Computer Science\nToront

In [13]:
def get_all_documents(
    client: QdrantClient,
    collection_name: str,
    batch_size: int = 100,
) -> list:
    """Return a list of all points (including payload) from the given collection.

    The Qdrant ``scroll`` API yields batches of points. We iterate until the
    iterator is exhausted and collect the points in a plain Python ``list``.

    Parameters
    ----------
    client: QdrantClient
        An instantiated Qdrant client.
    collection_name: str
        Name of the collection to read from.
    batch_size: int, optional
        Number of points to retrieve per request (default 100).
    """
    all_points = []
    for batch in client.scroll(
        collection_name=collection_name,
        limit=batch_size,
        with_payload=True,
    ):
        all_points.extend(batch.points)
    return all_points

documents = get_all_documents(client, "document_collection", batch_size=100)
print(f"🗂️  Total documents indexed: {len(documents)}")

# 3️⃣  Show a few samples (ID + stored payload, e.g., the original chunk text)
for point in documents[:5]:          # show first 5 records
    print("-" * 40)
    print(f"ID: {point.id}")
    # payload is a dict; most often it contains the original text under a key like "text"
    for key, value in point.payload.items():
        print(f"{key}: {value}")

AttributeError: 'list' object has no attribute 'points'

In [7]:
from langchain_openai import OpenAIEmbeddings
dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print(dense_embeddings)
print(type(dense_embeddings))
print(dense_embeddings.model)
print(dense_embeddings.model_kwargs)
print(dense_embeddings._invocation_params)

print(
    len(
        dense_embeddings.embed_documents(
            ["dummy_text"]
        )[0]
    )
)

client=<openai.resources.embeddings.Embeddings object at 0x13e6d0cd0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x13e6d11d0> model='text-embedding-3-large' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
<class 'langchain_openai.embeddings.base.OpenAIEmbeddings'>
text-embedding-3-large
{}
{'model': 'text-embedding-3-large'}
3072


In [6]:
dense_embeddings.embed_query("hello")

[-0.02459716796875,
 -0.007526397705078125,
 0.003971099853515625,
 0.006011962890625,
 0.005107879638671875,
 0.00018525123596191406,
 -0.006420135498046875,
 0.052459716796875,
 0.0096435546875,
 0.056488037109375,
 -0.00434112548828125,
 -0.0212249755859375,
 -0.0215911865234375,
 -0.01035308837890625,
 -0.0178680419921875,
 0.0201568603515625,
 0.024078369140625,
 0.00597381591796875,
 0.0151214599609375,
 -0.01189422607421875,
 0.01065826416015625,
 -0.00814056396484375,
 0.0011196136474609375,
 0.03509521484375,
 0.01004791259765625,
 0.0312347412109375,
 -0.0208587646484375,
 -0.0172271728515625,
 -0.0491943359375,
 -0.033782958984375,
 0.00083160400390625,
 0.026336669921875,
 0.0019407272338867188,
 0.0190887451171875,
 0.03564453125,
 0.007129669189453125,
 0.026641845703125,
 0.00438690185546875,
 -0.0239105224609375,
 0.02239990234375,
 0.026641845703125,
 0.00872802734375,
 -0.024444580078125,
 -0.0020904541015625,
 -0.00164794921875,
 -0.0018644332885742188,
 0.0122909545

## Create a collection

In [2]:
from qdrant_client.models import Distance, VectorParams


client.recreate_collection(
    collection_name="my_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)


/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/ipykernel_30273/753245820.py:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

## Add vectors

In [3]:
from qdrant_client.models import PointStruct

operation = client.upsert(
    collection_name="my_collection",
    points=[
        PointStruct(
            id=1,
            vector=[0.1] * 1536,
            payload={"text": "This is a sample document."}
        )
    ]
)


In [4]:
client.upsert(
    collection_name="my_collection",
    points=[
            PointStruct(
                id=1,
                vector=[0.2] * 1536,
                payload={"text": "This is a sample document."}
            ),
            PointStruct(
                id=2,
                vector=[0.3] * 1536,
                payload={"text": "This is another sample document."}
            ),
            PointStruct(
                id=3,
                vector=[0.4] * 1536,
                payload={"text": "This is a third sample document."}
            )
    ]
)

UpdateResult(operation_id=6, status=<UpdateStatus.COMPLETED: 'completed'>)

In [13]:
result =client.query_points(
    collection_name="my_collection",
    query=[0.1] * 1536,
    limit=5,
    with_payload=True
    # query_text  ="Hello world"
)

In [15]:
for point in result.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

ID: 3, Score: 1.0000015, Payload: {'text': 'This is a third sample document.'}
ID: 1, Score: 1.0000015, Payload: {'text': 'This is a sample document.'}
ID: 2, Score: 1.0000007, Payload: {'text': 'This is another sample document.'}


## BM25 vs dense search

BM25 is useful when exact words matter, such as product names, error codes, model numbers, or part numbers. Dense search is better when you care more about meaning than exact keywords.

In [20]:
from fastembed.sparse.sparse_text_embedding import SparseTextEmbedding
from qdrant_client.models import SparseVector

# 1) Dense search: semantic similarity with a vector query.
dense_hits = client.query_points(
    collection_name="my_collection",
    query=[0.1] * 1536,
    limit=3,
    with_payload=True,
)

print("Dense search results:")
for point in dense_hits.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

# 2) BM25 search: exact keyword matching on a sparse text collection.
client.set_sparse_model("Qdrant/bm25")

bm25_collection = "bm25_demo"
client.add(
    collection_name=bm25_collection,
    documents=[
        "Apple iPhone 15 Pro Max 256GB",
        "Samsung Galaxy S24 Ultra 256GB",
        "Apple MacBook Air M3",
    ],
)

bm25_model = SparseTextEmbedding("Qdrant/bm25")
bm25_query = list(bm25_model.query_embed("iphone 256gb"))[0]

bm25_hits = client.query_points(
    collection_name=bm25_collection,
    query=SparseVector(
        indices=bm25_query.indices.tolist(),
        values=bm25_query.values.tolist(),
    ),
    using=client.get_sparse_vector_field_name(),
    limit=3,
    with_payload=True,
)

print("\nBM25 search results:")
for point in bm25_hits.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

Dense search results:
ID: 3, Score: 1.0000015, Payload: {'text': 'This is a third sample document.'}
ID: 1, Score: 1.0000015, Payload: {'text': 'This is a sample document.'}
ID: 2, Score: 1.0000007, Payload: {'text': 'This is another sample document.'}

BM25 search results:
ID: 9dd20a04-519e-4336-a72c-9982c77fa0d6, Score: 2.4503899, Payload: {'document': 'Apple iPhone 15 Pro Max 256GB'}
ID: 2edc37e6-009c-4fa9-94e7-5f94ef5c3a03, Score: 2.4503899, Payload: {'document': 'Apple iPhone 15 Pro Max 256GB'}
ID: 9340239e-d193-4bf9-97b9-a92e2626fba2, Score: 0.73774153, Payload: {'document': 'Samsung Galaxy S24 Ultra 256GB'}


In [25]:
import time
from fastembed import TextEmbedding
from fastembed.sparse.sparse_text_embedding import SparseTextEmbedding
from qdrant_client.models import PointStruct, SparseVector, VectorParams, Distance

# Same dataset, two collections: one dense, one sparse/BM25.
comparison_docs = [
    {"id": 1, "text": "Apple iPhone 15 Pro Max 256GB"},
    {"id": 2, "text": "Samsung Galaxy S24 Ultra 256GB"},
    {"id": 3, "text": "Apple MacBook Air M3"},
    {"id": 4, "text": "Android phone with 256GB storage"},
]

queries = [
    ("iphone 256gb", 1),
    ("apple laptop m3", 3),
    ("galaxy ultra", 2),
]

# Dense collection
client.recreate_collection(
    collection_name="dense_compare",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

dense_model = TextEmbedding()

dense_points = []
for item in comparison_docs:
    dense_vector = list(dense_model.query_embed(item["text"]))[0].tolist()
    dense_points.append(
        PointStruct(
            id=item["id"],
            vector=dense_vector,
            payload={"text": item["text"]},
        )
    )

client.upsert(collection_name="dense_compare", points=dense_points)

# Sparse/BM25 collection
client.set_sparse_model("Qdrant/bm25")
client.recreate_collection(
    collection_name="bm25_compare",
    sparse_vectors_config={
        client.get_sparse_vector_field_name(): client.get_fastembed_sparse_vector_params()[
            client.get_sparse_vector_field_name()
        ]
    },
)

bm25_model = SparseTextEmbedding("Qdrant/bm25")

bm25_points = []
for item in comparison_docs:
    sparse_vector = list(bm25_model.query_embed(item["text"]))[0]
    bm25_points.append(
        PointStruct(
            id=item["id"],
            vector={
                client.get_sparse_vector_field_name(): SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                )
            },
            payload={"text": item["text"]},
        )
    )

client.upsert(collection_name="bm25_compare", points=bm25_points)

# Compare top-1 hit rate and elapsed time for the same queries.
def evaluate_collection(collection_name: str, query_builder, using: str | None = None):
    hits = 0
    start_time = time.perf_counter()

    for query_text, expected_id in queries:
        query_value = query_builder(query_text)
        response = client.query_points(
            collection_name=collection_name,
            query=query_value,
            using=using,
            limit=1,
            with_payload=True,
        )
        if response.points and response.points[0].id == expected_id:
            hits += 1

    elapsed_seconds = time.perf_counter() - start_time
    hit_rate = hits / len(queries)
    avg_ms = (elapsed_seconds / len(queries)) * 1000
    return hit_rate, elapsed_seconds, avg_ms

# Dense search
# This measures the full query flow, including embedding the query text.
dense_hit_rate, dense_elapsed, dense_avg_ms = evaluate_collection(
    "dense_compare",
    query_builder=lambda text: list(dense_model.query_embed(text))[0].tolist(),
    using=None,
)

# BM25 search
bm25_hit_rate, bm25_elapsed, bm25_avg_ms = evaluate_collection(
    "bm25_compare",
    query_builder=lambda text: SparseVector(
        indices=(bm25_vec := list(bm25_model.query_embed(text))[0]).indices.tolist(),
        values=bm25_vec.values.tolist(),
    ),
    using=client.get_sparse_vector_field_name(),
)

print(f"Dense top-1 hit rate: {dense_hit_rate:.2f}")
print(f"Dense total time:     {dense_elapsed:.4f} s")
print(f"Dense avg/query:      {dense_avg_ms:.2f} ms")
print(f"BM25 top-1 hit rate:   {bm25_hit_rate:.2f}")
print(f"BM25 total time:       {bm25_elapsed:.4f} s")
print(f"BM25 avg/query:        {bm25_avg_ms:.2f} ms")

/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/ipykernel_14659/614330401.py:21: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Dense top-1 hit rate: 1.00
Dense total time:     0.0177 s
Dense avg/query:      5.90 ms
BM25 top-1 hit rate:   1.00
BM25 total time:       0.0082 s
BM25 avg/query:        2.73 ms


/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/ipykernel_14659/614330401.py:43: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
